In [ ]:
import torch
from torchvision import transforms

from src.cnn.VisDroneCNN import VisDroneCNN
from src.dataset.Dataset import VisDrone
from src.dataset.collate import collate_fn

from src.detection_demo.VisualizePrediction import VisualizePrediction

### Add params

In [ ]:
model_weights_path = 'type/your/path/here'
img_h: int = 0
img_w: int = 0
conf_threshold: float = 0.7

### Transform data

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((img_h, img_w))
])

### Build model
Load model and make prediction for given images

In [ ]:
train_dev_data = "data/VisDrone_Dataset/VisDrone2019-DET-test-dev/images"
train_labels = "data/VisDrone_Dataset/VisDrone2019-DET-test-dev/labels"

visdrone_test = VisDrone(data_dir=train_dev_data, labels_dir=train_labels, transform=transform)
dataset = torch.utils.data.DataLoader(visdrone_test,
                                      batch_size=8,
                                      shuffle=True,
                                      num_workers=4,
                                      collate_fn=collate_fn,
                                      pin_memory=True,
                                      prefetch_factor=2)


cnn = VisDroneCNN(S=16, B_boxes=1, C=10)
cnn.load_state_dict(torch.load(model_weights_path))
device = 'cuda' if torch.cuda.is_available() else 'cpu'

images, labels = next(iter(dataset))

cnn.to(device)
images = images.to(device)

with torch.no_grad():
    output: torch.Tensor = cnn(images)

In [ ]:
visualize = VisualizePrediction(input_image_batch=images,
                                output_batch_prediction=output,
                                img_w=img_w, img_h=img_h,
                                conf_threshold=conf_threshold)

In [ ]:
visualize.show(4)